## Fundamentação Teórica: Sistemas de Recomendação por Fatoração de Matrizes

### 1. Introdução: O Desafio da Recomendação

Sistemas de recomendação são algoritmos projetados para sugerir itens relevantes aos usuários. Aplicações como Netflix (filmes), Spotify (músicas) e Amazon (produtos) dependem fundamentalmente desses sistemas para personalizar a experiência do usuário e impulsionar o engajamento.
O ponto de partida para a maioria dos sistemas de recomendação é a matriz de utilidade, que podemos chamar de $R$.

Nesta matriz:

As linhas representam os $m$ usuários do sistema.

As colunas representam os $n$ itens (filmes, no nosso caso).

A entrada $R_{ui}$ na linha $u$ e coluna $i$ contém a avaliação que o usuário $u$ deu ao item $i$.

O principal desafio é que essa matriz é extremamente esparsa. A maioria das células está vazia, pois um usuário típico avaliou apenas uma pequena fração do total de itens disponíveis. O objetivo do nosso sistema é, portanto, preencher as entradas vazias dessa matriz com previsões de notas que o usuário provavelmente daria, para então recomendar os itens com as maiores previsões.

### 2. A Solução: Fatoração de Matrizes

A Fatoração de Matrizes é uma técnica de Álgebra Linear poderosa, parte de uma classe de algoritmos chamada filtragem colaborativa. A ideia central é que existem fatores latentes (características ocultas) que determinam os gostos dos usuários e as propriedades dos itens.

Por exemplo, um fator latente para filmes pode ser o quanto ele se alinha ao gênero "comédia" versus "drama", enquanto outro pode ser o quanto ele é um "blockbuster" versus um "filme de arte". De forma similar, um usuário pode ter uma preferência maior ou menor por cada um desses fatores.

A Fatoração de Matrizes busca decompor a grande e esparsa matriz de utilidade $R$ (de dimensão $m \times n$) em duas matrizes menores e densas:

Matriz de Usuários ($P$): De dimensão $m \times k$, onde cada linha $p_u$ é um vetor que representa o perfil de gosto do usuário $u$ em termos dos $k$ fatores latentes.

Matriz de Itens ($Q$): De dimensão $n \times k$, onde cada linha $q_i$ é um vetor que representa as características do item $i$ nesses mesmos $k$ fatores latentes.

A aproximação da matriz de utilidade original é dada pelo produto dessas duas matrizes:
$$R \approx P \times Q^T$$

Onde $k$ é o número de fatores latentes, um hiperparâmetro que escolhemos. Geralmente, $k$ é significativamente menor que $m$ e $n$, o que torna essa representação muito eficiente.

### 3. Realizando Previsões

Com as matrizes $P$ e $Q$ em mãos, prever a avaliação que um usuário $u$ daria a um item $i$ (que ele ainda não viu) torna-se uma operação simples. A previsão, que chamaremos de $\hat{r}_{ui}$, é simplesmente o produto escalar do vetor do usuário $p_u$ com o vetor do item $q_i$:

$$\hat{r}_{ui} = p_u \cdot q_i = \sum_{j=1}^{k} p_{uj}q_{ij}$$

Intuitivamente, se um usuário tem uma forte preferência por um fator latente (por exemplo, "filmes de ficção científica") e um filme também tem uma pontuação alta nesse mesmo fator, o produto escalar será alto, resultando em uma previsão de nota elevada.

### 4. Como o Modelo "Aprende" as Matrizes P e Q?

Como a matriz $R$ original possui muitos valores ausentes, não podemos usar métodos de decomposição diretos como o SVD (Decomposição em Valores Singulares) tradicional. Em vez disso, o problema é tratado como uma tarefa de otimização.

O objetivo é encontrar as matrizes $P$ e $Q$ que minimizem o erro entre as notas previstas ($\hat{r}_{ui}$) e as notas reais ($r_{ui}$) que já conhecemos (as que não estão em branco na matriz $R$). Para isso, definimos uma função de custo (ou erro), geralmente o erro quadrático, e adicionamos termos de regularização para evitar sobreajuste (overfitting):

$$\text{Custo} = \sum_{(u,i) \in K} (r_{ui} - p_u \cdot q_i)^2 + \lambda(\|P\|^2 + \|Q\|^2)$$

Onde:

$K$ é o conjunto de pares $(u,i)$ para os quais a avaliação $r_{ui}$ é conhecida.

$\lambda$ é o parâmetro de regularização.

Para minimizar essa função de custo, são utilizados algoritmos de otimização iterativos, como o Gradiente

Descendente Estocástico (SGD) ou os Mínimos Quadrados Alternados (ALS), que ajustam gradualmente os valores de $P$ e $Q$ até que o erro seja o menor possível.


# PARTE 2: CÓDIGO

In [ ]:
# Força a instalação de uma versão do NumPy compatível com a scikit-surprise
%pip install 'numpy<2'

In [ ]:
%pip install scikit-surprise # Instalar essa biblioteca

In [ ]:
# Baixar e descompactar os dados
!wget http://files.grouplens.org/datasets/movielens/ml-100k.zip # Baixa ml-100k.zip, um arquivo da internet que contém o conjunto de dados (avaliação dos filmes)
!unzip ml-100k.zip # Descompacta o arquivo ".zip" que foi baixado, extraindo os arquivos de dados para o ambiente

In [ ]:
# Importações e carregamento dos dados
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

# Cria um dicionário para mapear o ID de cada filme ao seu respectivo título
movie_titles = {}
# Abre o arquivo 'u.item' que contém as informações dos filmes
with open('ml-100k/u.item', 'r', encoding='ISO-8859-1') as f: # Usa a tabela 'ISO-8859-1' para ler o arquivo, por causa de caracteres especiais nos nomes dos filmes
    for line in f: # Lê o arquivo linha por linha.
        parts = line.split('|') # Divide a linha pelo caractere '|' para separar as informações
        movie_titles[int(parts[0])] = parts[1] # Armazena no dicionário: a chave é o ID (inteiro) e o valor é o título

# Carregar o dataset de avaliações usando a biblioteca Surprise
reader = Reader(line_format='user item rating timestamp', sep='\t') # Define o formato do arquivo de avaliações: usuário, item, nota, timestamp
data = Dataset.load_from_file('ml-100k/u.data', reader=reader) # Carrega os dados de avaliações a partir do arquivo 'u.data' usando o formato definido

In [ ]:
#Treinamento do modelo
print("Iniciando o treinamento do modelo...")

# Constrói o conjunto de dados de treinamento que a biblioteca Surprise pode utilizar
trainset = data.build_full_trainset()

# Instancia o algoritmo SVD (Matrix Factorization) que é a implementação da Fatoração de Matrizes
algo = SVD()

# Treina o modelo utilizando o conjunto de dados de treinamento
algo.fit(trainset)

print("Treinamento concluído!")

In [ ]:
# Gerar recomendações para um usuário específico

user_id = '196' # Escolhe um ID de usuário para gerar recomendações
n_recommendations = 10 # Define o número de recomendações que será exibido

# Cria um conjunto (set) com os IDs dos filmes que o usuário JÁ avaliou, para não recomendá-los novamente
movies_rated_by_user = set([movie_id for (movie_id, rating) in trainset.ur[trainset.to_inner_uid(user_id)]])

print(f"\nCalculando recomendações para o usuário {user_id}...")

predictions = [] # Cria uma lista vazia para armazenar as previsões
for movie_id in trainset.all_items(): # Itera sobre TODOS os filmes existentes no conjunto de dados
    if movie_id not in movies_rated_by_user: # Verifica se o filme atual NÃO está na lista de filmes já avaliados pelo usuário

        raw_movie_id = trainset.to_raw_iid(movie_id) # Converte o ID interno do filme para o ID original (raw_id)

        predicted_rating = algo.predict(user_id, raw_movie_id).est # Usa o modelo treinado (.predict()) para prever a nota que o usuário daria ao filme
        predictions.append( (raw_movie_id, predicted_rating) ) # Adiciona uma tupla (ID do filme, nota prevista) à nossa lista de previsões

# Ordena a lista de previsões com base na nota prevista (do maior para o menor)
predictions.sort(key=lambda x: x[1], reverse=True)

# Exibe as 'n' melhores recomendações.
print(f"\nTop {n_recommendations} recomendações para o usuário {user_id}:")
for movie_id, estimated_rating in predictions[:n_recommendations]:
    # Imprime o título do filme (convertendo o ID para inteiro para buscar no dicionário) e sua nota prevista.
    print(f"- {movie_titles[int(movie_id)]}: Nota prevista de {estimated_rating:.2f}")